# Multi-Agent Orchestration Observability

**Before/After Demo:** Transform black-box multi-agent orchestration into transparent routing with observability.

## 🚀 Quick Start

1. **Configure Azure OpenAI credentials** in `.env` file:

   **Option A: API Key (Recommended for demos)**
   ```env
   AZURE_OPENAI_AUTH_MODE=key
   AZURE_OPENAI_API_KEY=<your-api-key>
   ```

   **Option B: Entra ID (Enterprise environments)**
   ```env
   AZURE_OPENAI_AUTH_MODE=entra
   # Ensure you're logged in to the correct tenant:
   # az login --tenant <your-tenant-id>
   ```

2. **Run Cell 1** to verify authentication setup
3. **Continue with the demo** - all cells should now work without authentication errors

---

In [1]:
# Cell 1: Setup — imports, env config (.env), logging, OpenTelemetry
import sys, os, json, logging
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(override=True)

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from azure.identity import DefaultAzureCredential, AzureCliCredential, get_bearer_token_provider

from utils.logging_config import setup_logging, log_routing_decision, log_self_reflection
from utils.telemetry import setup_telemetry

# Structured JSON logger
logger = setup_logging(level=logging.INFO)

# OpenTelemetry tracer
tracer = setup_telemetry()

# Azure OpenAI config
AZURE_OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_DEPLOYMENT = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')
AZURE_OPENAI_AUTH_MODE = os.environ.get('AZURE_OPENAI_AUTH_MODE', 'entra').lower()
AZURE_OPENAI_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY')
AZURE_TENANT_ID = os.environ.get('AZURE_TENANT_ID')

if AZURE_OPENAI_AUTH_MODE not in {'entra', 'key'}:
    raise ValueError(f'Unsupported AZURE_OPENAI_AUTH_MODE: {AZURE_OPENAI_AUTH_MODE}')

def create_kernel() -> Kernel:
    """Create a Semantic Kernel with Azure OpenAI configuration."""
    kernel = Kernel()
    service_kwargs = {
        'service_id': 'azure_openai',
        'deployment_name': AZURE_OPENAI_DEPLOYMENT,
        'endpoint': AZURE_OPENAI_ENDPOINT,
        'api_version': AZURE_OPENAI_API_VERSION,
    }

    if AZURE_OPENAI_AUTH_MODE == 'entra':
        # Use AzureCliCredential when AZURE_TENANT_ID is set so that the credential
        # explicitly targets the Azure OpenAI resource's tenant (avoids "Tenant
        # provided in token does not match resource token" errors).
        if AZURE_TENANT_ID:
            credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID)
        else:
            credential = DefaultAzureCredential()
        token_provider = get_bearer_token_provider(
            credential,
            'https://cognitiveservices.azure.com/.default',
        )
        service_kwargs['ad_token_provider'] = token_provider
    else:
        if not AZURE_OPENAI_API_KEY:
            raise ValueError('AZURE_OPENAI_API_KEY is required when AZURE_OPENAI_AUTH_MODE=key')
        service_kwargs['api_key'] = AZURE_OPENAI_API_KEY

    kernel.add_service(AzureChatCompletion(**service_kwargs))
    return kernel


def probe_auth() -> tuple[bool, str]:
    """Quick probe to verify credentials work BEFORE running expensive cells.

    Returns (ok, message).
    """
    if AZURE_OPENAI_AUTH_MODE == 'key':
        if AZURE_OPENAI_API_KEY:
            return True, 'API key configured'
        return False, 'AZURE_OPENAI_API_KEY missing in .env'

    # Entra ID: try to actually fetch a token
    try:
        if AZURE_TENANT_ID:
            cred = AzureCliCredential(tenant_id=AZURE_TENANT_ID)
        else:
            cred = DefaultAzureCredential()
        token = cred.get_token('https://cognitiveservices.azure.com/.default')
        if token and token.token:
            return True, f'Entra token acquired (expires in {(token.expires_on - int(__import__("time").time())) // 60} min)'
        return False, 'Empty token returned'
    except Exception as e:
        return False, f'{type(e).__name__}: {e}'


print('Setup complete')
print(f'  Endpoint:    {AZURE_OPENAI_ENDPOINT[:50]}...')
print(f'  Deployment:  {AZURE_OPENAI_DEPLOYMENT}')
print(f'  API version: {AZURE_OPENAI_API_VERSION}')
print(f'  Auth mode:   {AZURE_OPENAI_AUTH_MODE}')
print(f'  Tenant ID:   {AZURE_TENANT_ID or "(default)"}')
print(f'  SK version:  {__import__("semantic_kernel").__version__}')

# Probe credentials so failures are visible HERE, not deep in Section 1a's stack trace.
ok, msg = probe_auth()
print()
if ok:
    print(f'✅ Credential probe OK — {msg}')
else:
    print(f'❌ Credential probe FAILED — {msg}')
    print()
    print('How to fix:')
    if AZURE_OPENAI_AUTH_MODE == 'entra':
        tenant = AZURE_TENANT_ID or '<your-tenant-id>'
        print(f'  Option A (recommended): switch to API key auth')
        print(f'    Edit .env:')
        print(f'      AZURE_OPENAI_AUTH_MODE=key')
        print(f'      AZURE_OPENAI_API_KEY=<your-azure-openai-key>')
        print(f'    Then restart the kernel and re-run this cell.')
        print()
        print(f'  Option B: log in to the correct tenant, then re-run this cell:')
        print(f'    az login --tenant {tenant}')
    else:
        print(f'  Edit .env and set AZURE_OPENAI_API_KEY=<your-key>, then re-run this cell.')
    print()
    print('  Subsequent cells (Section 1a, 1b, 2, 3, 4) WILL FAIL until this is fixed.')


{"timestamp": "2026-04-20T01:17:37.200417+00:00", "level": "WARNING", "logger": "routing_observability.telemetry", "message": "azure-monitor-opentelemetry-exporter not installed. Using console exporter."}
Setup complete
  Endpoint:    https://dlstmvprtus-wingnut0310-ai.openai.azure.co...
  Deployment:  gpt-4o
  API version: 2024-12-01-preview
  Auth mode:   entra
  Tenant ID:   (default)
  SK version:  1.39.2

✅ Credential probe OK — Entra token acquired (expires in 45 min)


In [2]:
# Cell 2: Define 3 dummy agents + load mock KB data
from agents.common.recommend_agent import RecommendPlugin
from agents.common.search_agent import SearchPlugin
from agents.common.policy_agent import PolicyPlugin

with open('mock_data/products.json', encoding='utf-8') as f:
    PRODUCTS = json.load(f)
with open('mock_data/search_index.json', encoding='utf-8') as f:
    SEARCH_INDEX = json.load(f)
with open('mock_data/policies.json', encoding='utf-8') as f:
    POLICIES = json.load(f)

rp = RecommendPlugin()
sp = SearchPlugin()
pp = PolicyPlugin()

print(f'Products catalog: {len(PRODUCTS)} items')
for p in PRODUCTS:
    stock = 'in stock' if p['in_stock'] else 'out of stock'
    print(f'  {p["name"]} ({p["brand"]}) - {p["price"]:,}won | {stock}')
print(f'\nSearch index: {len(SEARCH_INDEX)} entries')
print(f'Policy docs: {len(POLICIES)} documents')
print()
print('Demo queries:')
print('  1. 건성 피부에 좋은 크림 추천해줘')
print('  2. 설화수 자음생 크림 가격 알려줘')
print('  3. 교환 환불 정책 알려줘')
print('  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?')


Products catalog: 5 items
  설화수 자음생 크림 (설화수) - 120,000won | in stock
  라네즈 워터뱅크 하이드로 크림 (라네즈) - 42,000won | in stock
  이니스프리 그린티 씨드 세럼 (이니스프리) - 29,000won | in stock
  헤라 블랙 쿠션 파운데이션 (헤라) - 55,000won | in stock
  마몽드 로즈워터 토너 (마몽드) - 18,000won | out of stock

Search index: 5 entries
Policy docs: 3 documents

Demo queries:
  1. 건성 피부에 좋은 크림 추천해줘
  2. 설화수 자음생 크림 가격 알려줘
  3. 교환 환불 정책 알려줘
  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?


---
# SECTION 1a: BEFORE — Generic Black Box

**Single system-prompt completion. NO plugins, NO function calling.**

3개 역할을 system prompt에 직접 기술하고, 단일 LLM completion으로 응답합니다.  
어떤 역할이 응답을 처리했는지 **전혀 알 수 없는** 완전한 black box입니다.

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [3]:
# Cell 4: Section 1a — Run black-box supervisor
from agents.before.blackbox_supervisor import run_blackbox_supervisor

before_kernel = create_kernel()

queries_1a = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1a: Generic Black Box')
print('(NO plugins, NO function calling)')
print('=' * 60)

for q in queries_1a:
    print(f'\nQuery: {q}')
    response = await run_blackbox_supervisor(q, before_kernel)
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> No way to know which role handled this query')
    print('-' * 60)


SECTION 1a: Generic Black Box
(NO plugins, NO function calling)

Query: 건성 피부에 좋은 크림 추천해줘
Response:
건성 피부에 적합한 크림으로 다음 제품들을 추천드립니다:

1. **아비브(Abib) 꿀한율크림** - 건조한 피부에 풍부한 보습을 제공하며 끈적임 없이 빠르게 흡수됩니다.
2. **세라비(CeraVe) 모이스처라이징 크림** - 세라마이드가 풍부하여 피부 장벽을 강화하고 오랜 시간 촉촉함을 유지합니다.
3. **라로슈포제(Laroche-Posay) 톨러리안 울트라 크림** - 민감한 건성 피부에도 적합하며 진정 효과를 제공합니다.
4. **바이오더마(Bioderma) 아토덤 크림** - 깊은 보습 효과와 함께 피부의 유수분 밸런스를 맞춰줍니다.

위 제품들은 보습력이 뛰어나고 자극 없이 사용할 수 있어 건성 피부에 추천드립니다. 피부 상태에 따라 맞는 제품을 선택해보세요!

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------------

Query: 설화수 자음생 크림 가격 알려줘
Response:
설화수 자음생크림의 가격은 제품 용량 및 판매처에 따라 다를 수 있습니다. 자음생크림 EX 60ml 기준, 공식 온라인 몰에서는 약 270,000원에 판매되고 있습니다. 다른 쇼핑몰이나 이벤트에 따라 가격이 다를 수 있으니 구매 전에 확인하시길 추천드립니다.

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------------

Query: 교환 환불 정책 알려줘
Response:
상품 교환 및 환불 정책은 다음과 같습니다:

1. **교환 및 환불 가능 

---
# SECTION 1b: BEFORE — OK Architecture Mirror

**고객 현재 아키텍처 재현:** `ChatCompletionAgent` + plugins + `invoke()`

OK의 `ok_sk_agent_flow` 구조를 mirror합니다:
- Supervisor agent가 3개 plugin을 가지고 있음
- `invoke()` 한 번에 routing + tool calling + 응답 생성이 모두 처리됨
- **Routing decision이 LLM call 안에 buried** → 별도 로깅 hook 없음

```
ChatCompletionAgent(kernel, supervisor_instructions)
    └── invoke(history)
        ├── LLM decides which plugin/tool to call   ◄── ROUTING HERE (no hook)
        ├── tool calls execute                       ◄── NO SEPARATE LOG
        └── response streams back                    ◄── COMBINED OUTPUT
```

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [4]:
# Cell 6: Section 1b — Run OK pattern supervisor
from agents.before.blackbox_ok_pattern import run_ok_pattern_supervisor

ok_kernel = create_kernel()

queries_1b = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1b: OK Architecture Mirror')
print('(ChatCompletionAgent + Plugins)')
print('=' * 60)

for q in queries_1b:
    print(f'\nQuery: {q}')
    # <<< ROUTING DECISION HAPPENS HERE — buried inside LLM call, no hook available >>>
    response = await run_ok_pattern_supervisor(q, ok_kernel)
    # <<< TOOL SELECTION — no separate logging, combined with response generation >>>
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> Tool calls happened but no routing-level visibility')
    # <<< This is why the customer cannot see which agent was selected or why >>>
    print('-' * 60)


SECTION 1b: OK Architecture Mirror
(ChatCompletionAgent + Plugins)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool RecommendPlugin-recommend_product",
    "context": {
        "trace_id": "0xa3e3b39289573b43bc3e577c6021ffdd",
        "span_id": "0x604b7186e8e5a2f8",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0dd5a5c47c229f5d",
    "start_time": "2026-04-20T01:17:55.383407Z",
    "end_time": "2026-04-20T01:17:55.383742Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "RecommendPlugin-recommend_product",
        "gen_ai.tool.call.id": "call_vWz08lewyR0Z9aOAovQrpIOt",
        "gen_ai.tool.description": "\ud53c\ubd80 \ud0c0\uc785\uc774\ub098 \uace0\ubbfc\uc5d0 \ub9de\ub294 \uc0c1\ud488\uc744 \ucd94\ucc9c\ud569\ub2c8\ub2e4. \uc608: '\uac74\uc131 \ud53c\ubd80\uc5d0 \uc88b\uc740 \ud06c\ub9bc \ucd94\ucc9c\ud574\uc918'"
    },
    "events": 

---
# SECTION 2: AFTER — Transparent Routing

**`AgentGroupChat` + `KernelFunctionSelectionStrategy`로 라우팅을 분리합니다.**

핵심 변화:
- **LLM call #1** (Selection): 어떤 agent를 선택할지 결정 (routing only)
- **`result_parser`** 가 선택 결과를 structured JSON으로 로깅
- **LLM call #2** (Response): 선택된 agent가 실제 응답 생성
- **Override hook**: 잘못된 라우팅을 커스텀 로직으로 교정 가능
- **OpenTelemetry** span으로 Application Insights에 trace 전송

| Routing Trace | Override | Logging Hook | OTel |
|:---:|:---:|:---:|:---:|
| ✅ agent, reason, confidence | ✅ Yes | ✅ JSON | ✅ App Insights |


In [5]:
# Cell 8: Section 2 — Transparent routing queries
from agents.after.transparent_routing import build_transparent_routing_chat

after_kernel = create_kernel()

queries_2 = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 2: Transparent Routing')
print('(AgentGroupChat + KernelFunctionSelectionStrategy)')
print('=' * 60)

for q in queries_2:
    print(f'\nQuery: {q}')
    chat, agents = build_transparent_routing_chat(
        kernel=after_kernel, logger=logger, tracer=tracer, query=q,
    )
    await chat.add_chat_message(message=q)
    async for response in chat.invoke():
        if response.content and response.name:
            print(f'\n[{response.name}]: {response.content[:500]}')
    print(f'\nRouting Trace: See structured JSON log above')
    print('-' * 60)


SECTION 2: Transparent Routing
(AgentGroupChat + KernelFunctionSelectionStrategy)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x740d852d159775ca5584983a7285a5e1",
        "span_id": "0xcccbbe7af611ed8c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-20T01:18:03.652406Z",
    "end_time": "2026-04-20T01:18:07.237428Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-20T01:18:07.2396

In [6]:
# Cell 9: Section 2 — Override demo with ambiguous query
ambiguous_query = '설화수 자음생 크림 건성 피부에 좋아?'

override_rules = {
    '설화수 자음생 크림 건성 피부에 좋아?': 'SearchAgent',
}

print('=' * 60)
print('SECTION 2: Override Demo — Ambiguous Query')
print('=' * 60)

# Without override
print(f'\n[Without Override]')
print(f'Query: {ambiguous_query}')
chat_no, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query,
)
await chat_no.add_chat_message(message=ambiguous_query)
async for resp in chat_no.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

# With override
print(f'\n[With Override] (override_applied: true)')
print(f'Query: {ambiguous_query}')
chat_ov, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query, override_rules=override_rules,
)
await chat_ov.add_chat_message(message=ambiguous_query)
async for resp in chat_ov.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

print(f'\nCompare logs: override_applied false -> true')
print('-' * 60)


SECTION 2: Override Demo — Ambiguous Query

[Without Override]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x585c3bbcf6899b7056e6f6bd11d59225",
        "span_id": "0xa242082ee9181d20",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-20T01:18:24.824574Z",
    "end_time": "2026-04-20T01:18:26.119384Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-20T01:18:26.120646+00:00", "leve

---
# SECTION 3: AFTER+ — Self-Reflection

**Section 2 + explicit `reflect()` function으로 응답 품질을 검증합니다.**

Flow:
1. **Routing** → AgentGroupChat이 agent 선택 (Section 2와 동일)
2. **KB Retrieval** → 선택된 agent가 mock KB에서 데이터 조회
3. **Reflection** → `reflect()` 함수가 별도 LLM call로 응답 품질 평가
4. **Re-route (REPLACE)** → 품질 미달 시 다른 agent로 교체, 원래 응답 폐기

핵심: `reflect()`는 **SK framework hook에 의존하지 않는** 명시적 Python function  
→ SK / MAF / LangGraph 어디서든 동일하게 작동 (framework-portable)

| Routing | Reflection | Re-route | Framework Portable |
|:---:|:---:|:---:|:---:|
| ✅ Full | ✅ Explicit | ✅ REPLACE | ✅ SK/MAF/LangGraph |


In [7]:
# Cell 11: Section 3 — Setup
from agents.after.self_reflection_routing import reflect, ReflectionResult
from agents.after.transparent_routing import (
    build_transparent_routing_chat,
    RECOMMEND_AGENT, SEARCH_AGENT, POLICY_AGENT,
)

reflection_kernel = create_kernel()

print('Section 3 setup complete')
print('  reflect() is a plain Python async function')
print('  -> Zero SK framework hook dependency')
print('  -> Portable to MAF / LangGraph without change')
print('  Pattern: route -> respond -> reflect -> (re-route if needed) -> final')


Section 3 setup complete
  reflect() is a plain Python async function
  -> Zero SK framework hook dependency
  -> Portable to MAF / LangGraph without change
  Pattern: route -> respond -> reflect -> (re-route if needed) -> final


In [8]:
# Cell 12: Section 3 — Two cases: KEEP vs REPLACE
from semantic_kernel.contents import ChatHistory

async def run_reflection_cycle(query: str, case_label: str, override_rules: dict[str, str] | None = None):
    """Run the full reflection cycle for a given query."""
    print(f'\n{"=" * 60}')
    print(f'[{case_label}]')
    print(f'Query: {query}')
    print('=' * 60)

    with tracer.start_as_current_span(f'section3_{case_label}') as span:
        span.set_attribute('query', query)

        # Step 1: Route via AgentGroupChat
        chat, agents_map = build_transparent_routing_chat(
            kernel=reflection_kernel, logger=logger, tracer=tracer,
            query=query, override_rules=override_rules,
        )
        await chat.add_chat_message(message=query)

        first_agent_name = None
        first_response = None

        async for response in chat.invoke():
            if response.content and response.name:
                first_agent_name = response.name
                first_response = response.content
                print(f'\n[Step 1 - Routing] Selected: {first_agent_name}')
                print(f'[{first_agent_name}]: {first_response[:400]}')

        # Step 2: KB retrieval (mock)
        if first_agent_name == RECOMMEND_AGENT:
            kb_raw = rp.recommend_product(query)
        elif first_agent_name == SEARCH_AGENT:
            kb_raw = sp.search_product(query)
        else:
            kb_raw = pp.lookup_policy(query)

        kb_results = json.loads(kb_raw)
        kb_items = kb_results.get('recommendations', kb_results.get('results', kb_results.get('policies', [])))
        print(f'\n[Step 2 - KB Retrieval] {len(kb_items)} items from {first_agent_name}')

        # Step 3: Self-reflection (rule-based pre-check + LLM fallback)
        print(f'\n[Step 3 - Reflection] Evaluating response quality...')
        reflection = await reflect(
            query=query,
            agent_name=first_agent_name or '',
            agent_response=first_response or '',
            kb_results=kb_items,
            kernel=reflection_kernel,
        )

        print(f'  intent_match:    {reflection.intent_match}')
        print(f'  sufficient_info: {reflection.sufficient_info}')
        print(f'  should_reroute:  {reflection.should_reroute}')
        print(f'  reroute_to:      {reflection.reroute_to}')
        print(f'  reason:          {reflection.reason}')

        if 'PARSE_FAILED' in reflection.reason:
            print(f'\n⚠️ WARNING: Reflection LLM returned malformed JSON.')
            print(f'  Raw output: {reflection.reason}')
            print(f'  Keeping original response as fallback.')

        # Step 4: Re-route if needed (REPLACE pattern)
        if reflection.should_reroute and reflection.reroute_to:
            reroute_agent_name = reflection.reroute_to
            print(f'\n[Step 4 - REPLACE] Re-routing: {first_agent_name} -> {reroute_agent_name}')

            reroute_agent = agents_map.get(reroute_agent_name)
            if reroute_agent:
                reroute_history = ChatHistory()
                reroute_history.add_user_message(query)
                final_response = None
                async for msg in reroute_agent.invoke(reroute_history):
                    text = str(msg)
                    if text:
                        final_response = text
                        print(f'[Final - {reroute_agent_name}]: {final_response[:400]}')
                action = 'REPLACE'
                final_agent = reroute_agent_name
            else:
                print(f'Agent {reroute_agent_name} not found, keeping original')
                final_response = first_response
                action = 'KEEP'
                final_agent = first_agent_name
        else:
            print(f'\n[Step 4] Reflection passed — keeping original response')
            final_response = first_response
            action = 'KEEP'
            final_agent = first_agent_name

        # Log self-reflection event
        log_self_reflection(
            logger,
            query=query,
            agent=first_agent_name or '',
            kb_results_count=len(kb_items),
            reflection={
                'intent_match': reflection.intent_match,
                'sufficient_info': reflection.sufficient_info,
                'should_reroute': reflection.should_reroute,
                'reroute_to': reflection.reroute_to,
                'reason': reflection.reason,
            },
            action=action,
            final_agent=final_agent,
        )
        span.set_attribute('reflection.action', action)
        span.set_attribute('reflection.final_agent', final_agent or '')

    # Full decision trace
    print('\n' + '-' * 60)
    print(f'Full Decision Trace [{case_label}]:')
    print(f'  1. Routing    -> {first_agent_name}')
    print(f'  2. KB         -> {len(kb_items)} items retrieved')
    print(f'  3. Reflection -> should_reroute={reflection.should_reroute}')
    print(f'  4. Action     -> {action} (final agent: {final_agent})')
    print('-' * 60)

# ═══════════════════════════════════════════════════════════
# Case 1: KEEP — 추천 의도만 있는 쿼리 (비즈니스 규칙 미해당)
# ═══════════════════════════════════════════════════════════
await run_reflection_cycle(
    query='설화수 자음생 크림 건성 피부에 좋아?',
    case_label='Case 1: KEEP',
)

# ═══════════════════════════════════════════════════════════
# Case 2: REPLACE — override로 RecommendAgent 강제 → 비즈니스 규칙 트리거
# ═══════════════════════════════════════════════════════════
query_replace = '설화수 자음생 크림 가격이랑 건성 피부에 맞는지 알려줘'
await run_reflection_cycle(
    query=query_replace,
    case_label='Case 2: REPLACE',
    override_rules={query_replace: 'RecommendAgent'},  # 데모용: 의도적 미스라우팅
)



[Case 1: KEEP]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x131bee40b8654106b242423e7c8d3e5c",
        "span_id": "0x9d7d29020b60cc3c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3bbaa12b023b1a1c",
    "start_time": "2026-04-20T01:18:36.680779Z",
    "end_time": "2026-04-20T01:18:39.969296Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-20T01:18:39.971275+00:00", "level": "INFO", "logger": "routing_

---
# SECTION 4: Latency Profiling

**SK 오케스트레이터 내부 스텝별 소요 시간을 측정하여 병목 구간을 데이터로 식별합니다.**

## 측정 대상

| # | 구간 | 측정 방법 | 의미 |
|---|------|---------|------|
| 0 | **Baseline** — 순수 LLM 직접 호출 | `chat_service.get_chat_message_contents()` | tool/agent 없이 LLM 자체 응답 시간 |
| 1 | **Agent Routing + Response** | `SelectionStrategy` + agent invoke | LLM call #1 + LLM call #2 |
| 2 | **AgentGroupChat.invoke() 전체** | invoke() 전후 시간 | SK orchestration 합계 |
| 3 | **KB Retrieval (Mock)** | plugin 호출 시간 | 로컬 JSON — 프로덕션은 MCP 네트워크 추가 |
| 4 | **Self-Reflection** | `reflect()` 함수 시간 | 규칙 기반 vs LLM 기반 분리 |
| 5 | **Re-route (REPLACE)** | 두 번째 agent invoke 시간 | 재라우팅 비용 |

## 측정하지 않는 것 (Mock 환경 한계)

- MCP 네트워크 왕복 시간 (Mock KB는 로컬 JSON 조회)
- 쿠버네티스 사이드카 통신
- APIM 로드밸런싱 레이턴시
- 프로덕션 부하 상황의 변동성

→ 고객이 자기 환경에서 동일 프로파일링 코드를 돌리면 이 부분까지 포함된 숫자가 나옵니다.


In [9]:
# Cell: Section 4 — Profiling functions
import time
from dataclasses import dataclass, field
from semantic_kernel.contents import ChatHistory
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings

# OTel InMemorySpanExporter — used to break down AgentGroupChat.invoke() internals.
# Two possible import paths depending on opentelemetry-sdk version.
try:
    from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
except ImportError:  # pragma: no cover — older OTel layout
    from opentelemetry.sdk.trace.export.in_memory import InMemorySpanExporter  # type: ignore

from opentelemetry import trace as _otel_trace
from opentelemetry.sdk.trace.export import SimpleSpanProcessor as _SimpleSpanProcessor


@dataclass
class StepProfile:
    """Per-step profiling record."""
    step_name: str
    duration_ms: float
    details: str = ""


@dataclass
class PipelineProfile:
    """Full pipeline profiling result."""
    query: str
    steps: list[StepProfile] = field(default_factory=list)
    total_ms: float = 0.0
    baseline_ms: float = 0.0
    overhead_ms: float = 0.0


# Module-level guard: register the InMemorySpanExporter exactly once.
# `provider.add_span_processor()` accumulates, so we must not register it on
# every profile_pipeline() call — that would multiply spans across runs.
_MEMORY_EXPORTER: InMemorySpanExporter | None = None


def _get_memory_exporter() -> InMemorySpanExporter:
    """Lazily attach an InMemorySpanExporter to the active TracerProvider."""
    global _MEMORY_EXPORTER
    if _MEMORY_EXPORTER is None:
        provider = _otel_trace.get_tracer_provider()
        exporter = InMemorySpanExporter()
        # SimpleSpanProcessor → spans become visible to the exporter as soon as
        # they end, which is what we want before reading them inside the cell.
        if hasattr(provider, "add_span_processor"):
            provider.add_span_processor(_SimpleSpanProcessor(exporter))
        _MEMORY_EXPORTER = exporter
    return _MEMORY_EXPORTER


async def measure_baseline(kernel, query: str) -> float:
    """Pure LLM direct call — baseline measurement (ms)."""
    chat_service = kernel.get_service('azure_openai')
    history = ChatHistory()
    history.add_system_message('당신은 뷰티 이커머스 챗봇입니다.')
    history.add_user_message(query)

    start = time.perf_counter()
    await chat_service.get_chat_message_contents(
        chat_history=history,
        settings=AzureChatPromptExecutionSettings(),
    )
    end = time.perf_counter()
    return (end - start) * 1000


def _build_invoke_breakdown(spans) -> list[StepProfile]:
    """Map collected OTel spans → indented StepProfile rows.

    SK auto-emits these span names:
      - `execute_tool routing_selection`     → routing LLM
      - `AutoFunctionInvocationLoop`         → agent LLM + tool calling
      - `execute_tool *Plugin-*`             → individual tool/plugin
      - `execute_tool termination`           → termination check LLM
    """
    breakdown: list[StepProfile] = []
    for s in sorted(spans, key=lambda x: x.start_time):
        duration_ms = (s.end_time - s.start_time) / 1e6  # ns → ms
        name = s.name
        attrs = s.attributes or {}

        if name == 'execute_tool routing_selection':
            breakdown.append(StepProfile(
                step_name='  ├ Routing LLM (agent 선택)',
                duration_ms=duration_ms,
                details=f'agent={attrs.get("routing.agent", "?")}',
            ))
        elif name == 'AutoFunctionInvocationLoop':
            funcs = str(attrs.get('sk.available_functions', ''))[:60]
            breakdown.append(StepProfile(
                step_name='  ├ Agent LLM + Tool Calling',
                duration_ms=duration_ms,
                details=f'functions={funcs}',
            ))
        elif name.startswith('execute_tool') and 'Plugin' in name:
            tool_name = name.replace('execute_tool ', '')
            breakdown.append(StepProfile(
                step_name=f'  │  └ Tool: {tool_name}',
                duration_ms=duration_ms,
                details='plugin 실행',
            ))
        elif name == 'execute_tool termination':
            breakdown.append(StepProfile(
                step_name='  └ Termination Check LLM',
                duration_ms=duration_ms,
                details='종료 판단',
            ))
    return breakdown


async def profile_pipeline(
    query: str,
    kernel,
    logger,
    tracer,
    override_rules: dict[str, str] | None = None,
) -> PipelineProfile:
    """Profile the full Section 3 pipeline step by step."""
    steps: list[StepProfile] = []
    total_start = time.perf_counter()

    # Step 0: Baseline (pure LLM)
    baseline_ms = await measure_baseline(kernel, query)

    # Prepare in-memory OTel exporter — clear any spans from previous runs so
    # the breakdown only reflects this invoke().
    memory_exporter = _get_memory_exporter()
    memory_exporter.clear()

    # Step 1+2: AgentGroupChat.invoke() — routing + response
    chat, agents_map = build_transparent_routing_chat(
        kernel=kernel, logger=logger, tracer=tracer,
        query=query, override_rules=override_rules,
    )
    await chat.add_chat_message(message=query)

    first_agent_name: str | None = None
    first_response: str | None = None
    routing_marked = False
    routing_end_ts: float | None = None

    invoke_start = time.perf_counter()
    async for response in chat.invoke():
        if response.content and response.name:
            if not routing_marked:
                routing_end_ts = time.perf_counter()
                routing_marked = True
            first_agent_name = response.name
            first_response = response.content
    invoke_end = time.perf_counter()

    if routing_end_ts is not None:
        steps.append(StepProfile(
            step_name='Agent Routing + Response (first message)',
            duration_ms=(routing_end_ts - invoke_start) * 1000,
            details=f'Selected: {first_agent_name}',
        ))

    steps.append(StepProfile(
        step_name='AgentGroupChat.invoke() 전체',
        duration_ms=(invoke_end - invoke_start) * 1000,
        details=f'Final agent: {first_agent_name}',
    ))

    # Collect invoke() internal spans → breakdown rows inserted right after
    # the "invoke() 전체" row above.
    try:
        if hasattr(memory_exporter, "force_flush"):
            memory_exporter.force_flush()
        collected_spans = memory_exporter.get_finished_spans()
    except Exception:  # pragma: no cover — exporter optional
        collected_spans = []

    invoke_breakdown = _build_invoke_breakdown(collected_spans)
    memory_exporter.clear()

    if invoke_breakdown:
        invoke_idx = next(
            (i for i, s in enumerate(steps) if 'invoke() 전체' in s.step_name),
            len(steps) - 1,
        )
        for j, bd in enumerate(invoke_breakdown):
            steps.insert(invoke_idx + 1 + j, bd)

    # Step 3: KB Retrieval (mock)
    kb_start = time.perf_counter()
    if first_agent_name == RECOMMEND_AGENT:
        kb_raw = rp.recommend_product(query)
    elif first_agent_name == SEARCH_AGENT:
        kb_raw = sp.search_product(query)
    else:
        kb_raw = pp.lookup_policy(query)
    kb_end = time.perf_counter()

    steps.append(StepProfile(
        step_name='KB Retrieval (Mock)',
        duration_ms=(kb_end - kb_start) * 1000,
        details='로컬 JSON — 프로덕션에서는 MCP 네트워크 시간 추가',
    ))

    # Step 4: Self-Reflection
    kb_results = json.loads(kb_raw)
    kb_items = kb_results.get(
        'recommendations',
        kb_results.get('results', kb_results.get('policies', [])),
    )

    reflect_start = time.perf_counter()
    reflection = await reflect(
        query=query,
        agent_name=first_agent_name or '',
        agent_response=first_response or '',
        kb_results=kb_items,
        kernel=kernel,
    )
    reflect_end = time.perf_counter()
    reflect_ms = (reflect_end - reflect_start) * 1000

    is_rule_based = reflection.should_reroute and '비즈니스 규칙' in reflection.reason
    reflect_type = '규칙 기반 (LLM 미호출)' if is_rule_based else 'LLM 기반'

    steps.append(StepProfile(
        step_name=f'Self-Reflection ({reflect_type})',
        duration_ms=reflect_ms,
        details=f'should_reroute={reflection.should_reroute}, reason={reflection.reason[:50]}',
    ))

    # Step 5: Re-route (REPLACE) — only if needed
    if reflection.should_reroute and reflection.reroute_to:
        reroute_agent = agents_map.get(reflection.reroute_to)
        if reroute_agent:
            reroute_start = time.perf_counter()
            reroute_history = ChatHistory()
            reroute_history.add_user_message(query)
            async for _ in reroute_agent.invoke(reroute_history):
                pass
            reroute_end = time.perf_counter()

            steps.append(StepProfile(
                step_name='Re-route (REPLACE)',
                duration_ms=(reroute_end - reroute_start) * 1000,
                details=f'{first_agent_name} → {reflection.reroute_to}',
            ))

    total_end = time.perf_counter()
    total_ms = (total_end - total_start) * 1000

    # Overhead = total - (baseline + non-overlapping pipeline steps)
    # Exclude:
    #   - "first message" row (overlaps with invoke() 전체)
    #   - breakdown rows starting with whitespace (children of invoke() 전체)
    pipeline_sum = sum(
        s.duration_ms for s in steps
        if 'first message' not in s.step_name and not s.step_name.startswith(' ')
    )
    overhead_ms = total_ms - baseline_ms - pipeline_sum

    return PipelineProfile(
        query=query,
        steps=steps,
        total_ms=total_ms,
        baseline_ms=baseline_ms,
        overhead_ms=overhead_ms,
    )


print('Section 4 profiling helpers loaded')
print('  - measure_baseline(): pure LLM call timing')
print('  - profile_pipeline(): full pipeline step-by-step timing')
print('  - InMemorySpanExporter: invoke() internal span breakdown')


Section 4 profiling helpers loaded
  - measure_baseline(): pure LLM call timing
  - profile_pipeline(): full pipeline step-by-step timing
  - InMemorySpanExporter: invoke() internal span breakdown


In [10]:
# Cell: Section 4 — Run profiling for KEEP and REPLACE cases
queries_profile = [
    ('KEEP', '설화수 자음생 크림 건성 피부에 좋아?', None),
    (
        'REPLACE',
        '설화수 자음생 크림 가격이랑 건성 피부에 맞는지 알려줘',
        {'설화수 자음생 크림 가격이랑 건성 피부에 맞는지 알려줘': 'RecommendAgent'},
    ),
]

for label, query, override in queries_profile:
    print(f'\n{"=" * 70}')
    print(f'[Profiling: {label}]')
    print(f'Query: {query}')
    print('=' * 70)

    profile = await profile_pipeline(
        query=query,
        kernel=reflection_kernel,
        logger=logger,
        tracer=tracer,
        override_rules=override,
    )

    # col_w widened so indented breakdown rows ("  ├ ...") render without
    # truncation.
    col_w = 55
    print(f'\n┌{"─" * col_w}┬{"─" * 12}┐')
    print(f'│ {"구간":<{col_w - 2}} │ {"소요시간":>10} │')
    print(f'├{"─" * col_w}┼{"─" * 12}┤')
    print(f'│ {"0. Baseline (순수 LLM 직접 호출)":<{col_w - 2}} │ {profile.baseline_ms:>8.0f}ms │')
    print(f'├{"─" * col_w}┼{"─" * 12}┤')

    for step in profile.steps:
        name = step.step_name[: col_w - 2]
        print(f'│ {name:<{col_w - 2}} │ {step.duration_ms:>8.0f}ms │')

    print(f'├{"─" * col_w}┼{"─" * 12}┤')
    print(f'│ {"SK 프레임워크 오버헤드 (측정 외 시간)":<{col_w - 2}} │ {profile.overhead_ms:>8.0f}ms │')
    print(f'├{"─" * col_w}┼{"─" * 12}┤')
    print(f'│ {"합계 (전체 파이프라인)":<{col_w - 2}} │ {profile.total_ms:>8.0f}ms │')
    print(f'└{"─" * col_w}┴{"─" * 12}┘')

    # 병목 분석 — exclude breakdown children (indented rows) so we report the
    # slowest top-level step rather than a sub-span.
    print(f'\n[병목 분석]')
    if profile.total_ms > 0:
        overhead_pct = (profile.total_ms - profile.baseline_ms) / profile.total_ms * 100
        print(f'  Baseline 대비 오버헤드: {overhead_pct:.1f}%')
    print(f'  → 순수 LLM: {profile.baseline_ms:.0f}ms, 파이프라인: {profile.total_ms:.0f}ms')

    top_level_steps = [s for s in profile.steps if not s.step_name.startswith(' ')]
    if top_level_steps:
        slowest = max(top_level_steps, key=lambda s: s.duration_ms)
        print(f'  가장 느린 구간: {slowest.step_name} ({slowest.duration_ms:.0f}ms)')
        print(f'    → {slowest.details}')

print('\n' + '=' * 70)
print('[고객 환경 적용 시 변경 포인트]')
print('  1. KB Retrieval → Mock JSON 대신 실제 MCP 호출 → 네트워크 시간 측정')
print('  2. Baseline → 동일 Azure OpenAI 엔드포인트 → APIM/리전 레이턴시 포함')
print('  3. Agent 수 → 실제 5~8개로 → tool description 부하 측정')
print('  4. 반복 측정 → 10회 이상 반복하여 P50/P95/P99 산출')
print('=' * 70)



[Profiling: KEEP]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x8d8da73e0092b03754acc6db8f79844b",
        "span_id": "0xd7810f1808d8e2aa",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-20T01:19:09.100366Z",
    "end_time": "2026-04-20T01:19:10.404984Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-20T01:19:10.405907+00:00", "level": "INFO", "logger": "routing_observability

---
# SECTION 5 — Routing 변경 없이 Latency 튜닝

Section 4에서 baseline pipeline은 LLM 호출이 누적되어 ~10초 가까이 소요됩니다.
**모델 변경 없이 (gpt-4o 유지)** 적용 가능한 3가지 튜닝을 같은 `profile_pipeline()` 흐름에서 측정해 비교합니다.

| 튜닝 | 효과 | Trade-off |
|------|------|-----------|
| `max_tokens` 제한 (라우팅/종료 50, 응답 250) | 응답 토큰 생성 시간 단축 | 답변이 잘릴 수 있음 |
| Agent별 Plugin 격리 (각 agent는 자기 plugin만) | tool selection 단순화 → 결정 빠름 | 에이전트 간 도구 공유 불가 |
| Termination LLM 제거 → `DefaultTerminationStrategy(maximum_iterations=1)` | LLM 호출 1회 제거 (~600ms+) | Multi-turn 협업 시나리오 부적합 |

> **주의**: 모델 다운그레이드(예: gpt-4o-mini)는 의도적으로 제외 — 동일 모델 유지가 고객 제약일 때 사용 가능한 튜닝만 시연.
> Streaming / Prompt caching 은 별도 Future Work.


In [11]:
# Cell: Section 5 — Optimized pipeline builder + profiling
from semantic_kernel.agents import AgentGroupChat, ChatCompletionAgent
from semantic_kernel.agents.strategies import (
    DefaultTerminationStrategy,
    KernelFunctionSelectionStrategy,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.connectors.ai.open_ai import AzureChatPromptExecutionSettings
from semantic_kernel.contents import ChatHistoryTruncationReducer
from semantic_kernel.functions import KernelArguments, KernelFunctionFromPrompt

from agents.after.transparent_routing import (
    POLICY_AGENT, RECOMMEND_AGENT, SEARCH_AGENT,
    SELECTION_PROMPT, create_result_parser,
)
from agents.common.policy_agent import PolicyPlugin
from agents.common.recommend_agent import RecommendPlugin
from agents.common.search_agent import SearchPlugin

OPTIMIZED_RESPONSE_MAX_TOKENS = 250
OPTIMIZED_ROUTING_MAX_TOKENS = 50


def build_optimized_chat(
    *, kernel, logger, tracer, query: str,
    override_rules: dict[str, str] | None = None,
):
    """3가지 튜닝 적용된 AgentGroupChat:
       1) Plugin 격리 (각 agent는 자기 plugin만)
       2) max_tokens 제한 (응답 250, 라우팅 50)
       3) Termination LLM 제거 → DefaultTerminationStrategy(1)
    """
    response_settings = AzureChatPromptExecutionSettings(
        max_tokens=OPTIMIZED_RESPONSE_MAX_TOKENS,
        function_choice_behavior=FunctionChoiceBehavior.Auto(),
    )

    # 1) + 2) — 각 agent별 own plugin + max_tokens 제한
    recommend_agent = ChatCompletionAgent(
        kernel=kernel, name=RECOMMEND_AGENT,
        instructions='당신은 뷰티 상품 추천 전문가입니다. 한국어로 간결히 답변하세요.',
        plugins=[RecommendPlugin()],
        arguments=KernelArguments(settings=response_settings),
    )
    search_agent = ChatCompletionAgent(
        kernel=kernel, name=SEARCH_AGENT,
        instructions='당신은 뷰티 상품 검색 전문가입니다. 한국어로 간결히 답변하세요.',
        plugins=[SearchPlugin()],
        arguments=KernelArguments(settings=response_settings),
    )
    policy_agent = ChatCompletionAgent(
        kernel=kernel, name=POLICY_AGENT,
        instructions='당신은 뷰티 커머스 정책 안내 전문가입니다. 한국어로 간결히 답변하세요.',
        plugins=[PolicyPlugin()],
        arguments=KernelArguments(settings=response_settings),
    )
    agents_map = {
        RECOMMEND_AGENT: recommend_agent,
        SEARCH_AGENT: search_agent,
        POLICY_AGENT: policy_agent,
    }

    # 라우팅 LLM도 max_tokens 제한
    routing_settings = AzureChatPromptExecutionSettings(
        max_tokens=OPTIMIZED_ROUTING_MAX_TOKENS,
    )
    selection_function = KernelFunctionFromPrompt(
        function_name='routing_selection',
        prompt=SELECTION_PROMPT,
        prompt_execution_settings={'default': routing_settings},
    )
    selection_function.prompt_template.allow_dangerously_set_content = True

    # 3) Termination LLM 제거 — single iteration only
    chat = AgentGroupChat(
        agents=list(agents_map.values()),
        selection_strategy=KernelFunctionSelectionStrategy(
            function=selection_function,
            kernel=kernel,
            result_parser=create_result_parser(logger, tracer, query, override_rules),
            history_variable_name='_history_',
            history_reducer=ChatHistoryTruncationReducer(target_count=1),
        ),
        termination_strategy=DefaultTerminationStrategy(maximum_iterations=1),
    )
    return chat, agents_map


async def profile_optimized_pipeline(
    query: str, kernel, logger, tracer,
    override_rules: dict[str, str] | None = None,
) -> PipelineProfile:
    """Section 4 profile_pipeline()와 동일한 측정 흐름 — chat builder만 교체."""
    steps: list[StepProfile] = []
    total_start = time.perf_counter()

    baseline_ms = await measure_baseline(kernel, query)

    memory_exporter = _get_memory_exporter()
    memory_exporter.clear()

    chat, _agents_map = build_optimized_chat(
        kernel=kernel, logger=logger, tracer=tracer,
        query=query, override_rules=override_rules,
    )
    await chat.add_chat_message(message=query)

    first_agent_name: str | None = None
    invoke_start = time.perf_counter()
    async for response in chat.invoke():
        if response.content and response.name:
            first_agent_name = response.name
    invoke_end = time.perf_counter()

    steps.append(StepProfile(
        step_name='AgentGroupChat.invoke() 전체 (Optimized)',
        duration_ms=(invoke_end - invoke_start) * 1000,
        details=f'Final agent: {first_agent_name}',
    ))

    try:
        if hasattr(memory_exporter, 'force_flush'):
            memory_exporter.force_flush()
        collected_spans = memory_exporter.get_finished_spans()
    except Exception:
        collected_spans = []

    invoke_breakdown = _build_invoke_breakdown(collected_spans)
    memory_exporter.clear()
    for j, bd in enumerate(invoke_breakdown):
        steps.insert(1 + j, bd)

    total_end = time.perf_counter()
    total_ms = (total_end - total_start) * 1000
    pipeline_sum = sum(
        s.duration_ms for s in steps if not s.step_name.startswith(' ')
    )
    overhead_ms = total_ms - baseline_ms - pipeline_sum

    return PipelineProfile(
        query=query, steps=steps,
        total_ms=total_ms, baseline_ms=baseline_ms, overhead_ms=overhead_ms,
    )


# Run optimized profiling on the same KEEP query as Section 4 for fair comparison
optimized_query = queries_profile[0][1]  # KEEP query
print('=' * 70)
print('[Section 5: Optimized Pipeline]')
print(f'Query: {optimized_query}')
print('  Tunings: max_tokens(250/50) + Plugin 격리 + DefaultTerminationStrategy(1)')
print('=' * 70)

optimized_profile = await profile_optimized_pipeline(
    query=optimized_query,
    kernel=reflection_kernel,
    logger=logger,
    tracer=tracer,
)

col_w = 55
print(f'\n┌{"─" * col_w}┬{"─" * 12}┐')
print(f'│ {"구간":<{col_w - 2}} │ {"소요시간":>10} │')
print(f'├{"─" * col_w}┼{"─" * 12}┤')
print(f'│ {"0. Baseline (순수 LLM)":<{col_w - 2}} │ {optimized_profile.baseline_ms:>8.0f}ms │')
print(f'├{"─" * col_w}┼{"─" * 12}┤')
for step in optimized_profile.steps:
    name = step.step_name[: col_w - 2]
    print(f'│ {name:<{col_w - 2}} │ {step.duration_ms:>8.0f}ms │')
print(f'├{"─" * col_w}┼{"─" * 12}┤')
print(f'│ {"SK 오버헤드":<{col_w - 2}} │ {optimized_profile.overhead_ms:>8.0f}ms │')
print(f'├{"─" * col_w}┼{"─" * 12}┤')
print(f'│ {"합계":<{col_w - 2}} │ {optimized_profile.total_ms:>8.0f}ms │')
print(f'└{"─" * col_w}┴{"─" * 12}┘')


[Section 5: Optimized Pipeline]
Query: 설화수 자음생 크림 건성 피부에 좋아?
  Tunings: max_tokens(250/50) + Plugin 격리 + DefaultTerminationStrategy(1)
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0xb0746342eb8714fbb5a63342f4588f6f",
        "span_id": "0x1a92f0bfac97927e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-20T01:19:41.656803Z",
    "end_time": "2026-04-20T01:19:42.705345Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp"

In [15]:
# Cell: Section 5 — Before/After comparison (Baseline vs Optimized)
# Re-run baseline pipeline on the SAME query for fair side-by-side comparison.
print('=' * 70)
print('[Comparison: Section 4 Baseline vs Section 5 Optimized]')
print(f'Query: {optimized_query}')
print('=' * 70)

baseline_profile = await profile_pipeline(
    query=optimized_query,
    kernel=reflection_kernel,
    logger=logger,
    tracer=tracer,
)


def _invoke_total(p: PipelineProfile) -> float:
    for s in p.steps:
        if 'invoke() 전체' in s.step_name:
            return s.duration_ms
    return 0.0


b_baseline = baseline_profile.baseline_ms
o_baseline = optimized_profile.baseline_ms
b_invoke = _invoke_total(baseline_profile)
o_invoke = _invoke_total(optimized_profile)
b_total = baseline_profile.total_ms
o_total = optimized_profile.total_ms

cw = 36
print(f'\n┌{"─" * cw}┬{"─" * 12}┬{"─" * 12}┬{"─" * 8}┐')
print(f'│ {"구간":<{cw - 2}} │ {"Baseline":>10} │ {"Optimized":>10} │ {"Δ":>6} │')
print(f'├{"─" * cw}┼{"─" * 12}┼{"─" * 12}┼{"─" * 8}┤')


def _row(label: str, b: float, o: float) -> None:
    delta = o - b
    sign = '↓' if delta < 0 else ('↑' if delta > 0 else '=')
    pct = (delta / b * 100) if b else 0.0
    print(
        f'│ {label:<{cw - 2}} │ {b:>8.0f}ms │ {o:>8.0f}ms │'
        f' {sign}{abs(pct):>4.0f}% │'
    )


_row('Baseline (순수 LLM, 변화 없음)', b_baseline, o_baseline)
_row('AgentGroupChat.invoke() 전체', b_invoke, o_invoke)
_row('TOTAL pipeline', b_total, o_total)
print(f'└{"─" * cw}┴{"─" * 12}┴{"─" * 12}┴{"─" * 8}┘')

saved_ms = b_total - o_total
saved_pct = saved_ms / b_total * 100 if b_total else 0.0
invoke_saved_pct = (b_invoke - o_invoke) / b_invoke * 100 if b_invoke else 0.0

print(f'\n[결론]')
print(f'  invoke() 절감: {b_invoke - o_invoke:>6.0f}ms  ({invoke_saved_pct:>5.1f}%)')
print(f'  TOTAL  절감: {saved_ms:>6.0f}ms  ({saved_pct:>5.1f}%)')
print(f'  → 모델 변경 없이 (gpt-4o 유지) 달성')
print()
print('[적용된 튜닝]')
print('  1. max_tokens 제한 (응답 250 / 라우팅 50)')
print('  2. Plugin 격리 (각 agent → 자기 plugin만)')
print('  3. Termination LLM 제거 (DefaultTerminationStrategy)')
print()
print('[Future Work — 본 데모 제외]')
print('  • Streaming: 체감 latency만 개선, total 동일')
print('  • Prompt caching: 2회차+ ~30% 절감, SK settings 호환성 검증 필요')
print('  • Model 다운그레이드: gpt-4o-mini → 추가 ~50% 가능 (품질 trade-off)')


[Comparison: Section 4 Baseline vs Section 5 Optimized]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0xbc31fe9a750c1b86247ba6cb6ed5a659",
        "span_id": "0xc77cab1ab980f7d6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-20T01:31:02.032894Z",
    "end_time": "2026-04-20T01:31:03.104765Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-20T01:31:03.105539+00:00", "level": "IN

---
# SUMMARY

## Before vs After

| 항목 | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ | 4: Profiling | 5: Tuning |
|------|:---:|:---:|:---:|:---:|:---:|:---:|
| **Routing 분리** | ❌ | ❌ | ✅ | ✅ | ✅ | ✅ |
| **Routing Trace** | ❌ | ❌ | ✅ agent, reason, confidence | ✅ + reflection | ✅ + timing | ✅ |
| **Override Hook** | ❌ | ❌ | ✅ result_parser | ✅ | ✅ | ✅ |
| **Self-Reflection** | ❌ | ❌ | ❌ | ✅ explicit reflect() | ✅ | — |
| **Re-route (REPLACE)** | ❌ | ❌ | ❌ | ✅ | ✅ | — |
| **OTel Tracing** | ❌ | ❌ | ✅ | ✅ | ✅ + span 분해 | ✅ |
| **Latency Profiling** | ❌ | ❌ | ❌ | ❌ | ✅ 구간별 ms | ✅ before/after |
| **Latency 튜닝** | — | — | — | — | — | ✅ max_tokens + 격리 + DefaultTermination |
| **Framework Portable** | — | — | SK only | ✅ SK/MAF/LangGraph | ✅ | ✅ |

## Key Takeaways

1. **SK는 이미 투명한 라우팅을 공식 지원합니다** — `KernelFunctionSelectionStrategy`를 활용하면 기존 코드 구조를 크게 변경하지 않고도 라우팅 가시성을 확보할 수 있습니다.
2. **라우팅과 응답 생성을 분리하면 운영이 쉬워집니다** — 로깅, 오버라이드, 모니터링이 자연스럽게 따라옵니다.
3. **Self-reflection은 framework에 종속되지 않습니다** — explicit function으로 구현하면 어디로든 이식 가능합니다.
4. **SK 프레임워크 자체 오버헤드는 ~10ms 미만** — 병목은 LLM call 횟수이며, OTel span 분해로 정확한 구간별 진단이 가능합니다.
5. **모델 변경 없이도 latency를 줄일 수 있습니다** — `max_tokens` 제한, plugin 격리, Termination LLM 제거 3가지로 invoke() 시간을 의미 있게 단축 (Section 5 비교표 참조).


In [13]:
# Cell 14: Summary — comparison table + App Insights KQL
from IPython.display import display, Markdown

comparison = """
## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ | 4: Profiling | 5: Tuning |
|---------|:---:|:---:|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + reflect [+ re-route]) | 측정 대상 | 2 (termination 제거) |
| Routing Visible | No | No | Yes | Yes | Yes + timing | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser | result_parser | result_parser |
| Quality Check | No | No | No | reflect() | reflect() | — |
| Re-route on Failure | No | No | No | REPLACE | REPLACE | — |
| OTel / App Insights | No | No | Yes | Yes | Yes + span 분해 | Yes |
| Latency Breakdown | No | No | No | No | ✅ 구간별 ms | ✅ before/after |
| Latency Tuning | — | — | — | — | — | max_tokens + 격리 + DefaultTermination |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() | + InMemorySpanExporter | + DefaultTerminationStrategy |
"""
display(Markdown(comparison))

kql = """
## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```
"""
display(Markdown(kql))

print('\nDemo complete!')



## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ | 4: Profiling | 5: Tuning |
|---------|:---:|:---:|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + reflect [+ re-route]) | 측정 대상 | 2 (termination 제거) |
| Routing Visible | No | No | Yes | Yes | Yes + timing | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser | result_parser | result_parser |
| Quality Check | No | No | No | reflect() | reflect() | — |
| Re-route on Failure | No | No | No | REPLACE | REPLACE | — |
| OTel / App Insights | No | No | Yes | Yes | Yes + span 분해 | Yes |
| Latency Breakdown | No | No | No | No | ✅ 구간별 ms | ✅ before/after |
| Latency Tuning | — | — | — | — | — | max_tokens + 격리 + DefaultTermination |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() | + InMemorySpanExporter | + DefaultTerminationStrategy |



## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```



Demo complete!


In [14]:
#